# 03 - Grade a prediction and compare to the baseline

This notebook walks the prediction contract end to end: build a schema-valid
`ParserOutput`, validate it, then load the bundled baseline scores to compare
against. The full grading run (`doc-bench --dataset ... --predictions ...`) is
a CLI step shown at the end.

## 1. Build a schema-valid prediction

In [ ]:
import json
from importlib.resources import files
import jsonschema

schema = json.loads((files('doc_bench') / 'fixtures' / 'parser_output.schema.json').read_text())

doc_id = '01030000000001'
text = 'The quick brown fox jumps over the lazy dog.'
prediction = {
    'schema_version': '1.0.0',
    'parser_version': 'notebook-demo-1.0.0',
    'parsed_at': '2026-06-04T00:00:00Z',
    'source': {
        'doc_id': doc_id, 'filename': f'{doc_id}.pdf', 'mime_type': 'application/pdf',
        'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855',
        'page_count': 1, 'language': 'en',
    },
    'pages': [{'page_index': 0, 'width': 612, 'height': 792, 'rotation': 0}],
    'elements': [{
        'element_id': f'{doc_id}_paragraph_000', 'type': 'paragraph', 'page_index': 0,
        'char_span': [0, len(text)], 'text': text, 'content': {'kind': 'text'},
        'bbox': {'x0': 72.0, 'y0': 110.0, 'x1': 540.0, 'y1': 128.0},
    }],
    'warnings': [],
}
jsonschema.validate(prediction, schema)
print('prediction is schema-valid')

## 2. Load the bundled baseline scores

In [ ]:
metrics = ['nid', 'teds', 'mhs', 'ard', 'bleu', 'meteor']
for name, fn in [('DP-Bench', 'dpbench_results.json'),
                 ('OmniDocBench', 'omnidocbench_results.json'),
                 ('ATO-Bench', 'ato_bench_results.json')]:
    avg = json.loads((files('doc_bench') / 'fixtures' / fn).read_text())['averages']
    print(name, {m: avg.get(m) for m in metrics})

## 3. Run a full grading pass (CLI)

Grading against ground truth is a CLI step. Export documents, write one
`<doc_id>.json` per document, then grade:

```bash
doc-bench-dump-dataset --dataset dp_bench --output ./pdfs --limit 5
# ... your parser writes ./predictions/<doc_id>.json ...
doc-bench --dataset dp_bench --predictions ./predictions --output-dir ./results
```

Results: a per-document CSV, a summary JSON with averages, and a rejected CSV.
Compare your summary averages against the baselines printed above.